# 평가 데이터 생성 및 비교 전체 코드
- Vector DB를 불러와서 Base 모델과 RAG 모델의 응답을 비교하고 CSV로 저장하는 전체 코드

In [ ]:
import os
import json
import pandas as pd
import getpass
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from tqdm import tqdm  # 진행률 표시를 위한 라이브러리

# ==========================================
# 1. 초기 설정 및 보안 인증
# ==========================================
MY_API_KEY = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)

# 경로 설정 (evaluation 폴더 기준 상위 폴더의 data 참조)
CHROMA_PATH = "../data/test/card_vector_db"

# Embedding 함수 설정 (기존 인덱싱 때와 동일해야 함)
ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# Vector DB 연결 (기존 컬렉션 불러오기)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name="card_separated_collection",
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 모델 응답 함수 정의
# ==========================================

# (1) Base GPT-3.5-turbo 응답 함수 (RAG 미적용)
def get_base_model_response(query):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "당신은 유능한 금융 상품 추천 전문가입니다. 아는 지식 내에서 답변해주세요."},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# (2) RAG 구성 후 응답 함수 (기존 로직 유지)
def get_rag_model_response(query, persona_dict, target_card_type=None):
    persona_traits = persona_dict['traits']
    search_query = f"{persona_traits} {query}"
    
    # DB 검색
    if target_card_type in ["체크", "신용"]:
        results = collection.query(
            query_texts=[search_query], 
            n_results=15, # 평가 효율을 위해 결과 수 조정 가능
            where={"card_type": target_card_type}
        )
    else:
        results = collection.query(query_texts=[search_query], n_results=15)

    raw_documents = results['documents'][0]
    available_card_names = set()
    card_context_map = {}
    
    for doc in raw_documents:
        first_line = doc.split("\n")[0]
        card_title = first_line.split("카드명: ")[1].split("|")[0].strip()
        available_card_names.add(card_title)
        if card_title not in card_context_map: card_context_map[card_title] = []
        card_context_map[card_title].append(doc)

    verified_list_str = ", ".join(list(available_card_names))
    formatted_context = ""
    for title, contents in card_context_map.items():
        formatted_context += f"\n### 카드 데이터: {title} ###\n" + "\n".join(contents) + "\n"

    system_prompt = f"""
    당신은 금융 상품 추천 전문가입니다. 반드시 [검증된 카드 리스트] 내의 카드만 추천하십시오.
    [검증된 카드 리스트]: {verified_list_str}
    [상세 데이터]: {formatted_context}
    [사용자 페르소나]: {persona_dict['name']} ({persona_traits})
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# ==========================================
# 3. 테스트 시나리오 설정
# ==========================================
test_scenarios = [
    {"persona": {"name": "20대 자기계발생", "traits": "무실적 카드 선호, 대중교통 및 학원 이용 잦음"}, "query": "전월 실적 부담 없으면서 대중교통, 학원 혜택이 좋은 체크카드 알려줘.", "target": "체크"},
    {"persona": {"name": "20대 사회초년생 자취형", "traits": "배달·카페·온라인 쇼핑 비중 높음"}, "query": "배달 음식과 카페, 온라인 쇼핑에서 자주 쓰기 좋은 체크카드 추천해줘.", "target": "체크"},
    {"persona": {"name": "20대 외출 활동형", "traits": "외식과 카페 이용 빈도 높음, 문화생활"}, "query": "카페와 외식 위주로 자주 쓰기 좋은 체크카드를 알려줘.", "target": "체크"},
    {"persona": {"name": "30대 출퇴근 직장인", "traits": "대중교통 출퇴근, 점심 외식과 커피 지출"}, "query": "출퇴근 교통비랑 회사 근처 카페 혜택이 집중된 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 자차 보유 직장인", "traits": "자차 출퇴근, 주유비 지출 큼"}, "query": "주유 혜택이 크고 자차 이용자에게 도움이 되는 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 여행 중심형", "traits": "해외 여행 및 직구 빈번, 항공권·숙박"}, "query": "해외 결제 수수료 혜택이 좋고 여행 시 활용하기 좋은 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 건강관리 소비형", "traits": "병원, 약국, 헬스장 이용 빈도 높음"}, "query": "병원이나 헬스장 지출에 혜택을 받을 수 있는 신용카드를 알려줘.", "target": "신용"},
    {"persona": {"name": "40대 가족 생활형", "traits": "대형마트 장보기, 생활비와 고정비 관리"}, "query": "대형마트 장보기랑 생활비 지출에 혜택이 좋은 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "40대 교육비 집중형", "traits": "자녀 학원비 결제 비중 큼"}, "query": "자녀 학원비 결제 시 혜택을 받을 수 있는 신용카드를 알려줘.", "target": "신용"},
    {"persona": {"name": "디지털 구독 소비형", "traits": "OTT, 온라인 쇼핑, 구독 서비스"}, "query": "온라인 결제나 구독 서비스에서 할인/적립이 있는 체크카드 추천해줘.", "target": "체크"}
]

# ==========================================
# 4. 평가 실행 및 CSV 저장
# ==========================================
ITERATIONS = 10 # 페르소나별 10회 반복 (총 100회)
evaluation_data = []

print(f"🚀 테스트 시작 (총 {len(test_scenarios) * ITERATIONS}건)")

for i, sc in enumerate(test_scenarios):
    print(f"📍 [{i+1}/10] {sc['persona']['name']} 시나리오 진행 중...")
    
    for run in tqdm(range(1, ITERATIONS + 1), desc="반복 회차"):
        # 응답 생성
        base_ans = get_base_model_response(sc['query'])
        rag_ans = get_rag_model_response(sc['query'], sc['persona'], target_card_type=sc['target'])
        
        evaluation_data.append({
            "시나리오_ID": i + 1,
            "회차": run,
            "페르소나": sc['persona']['name'],
            "요청_카드유형": sc['target'],
            "질문": sc['query'],
            "Base_GPT_응답": base_ans,
            "RAG_시스템_응답": rag_ans,
            "Human Evaluation": "",    # 직접 기입용
            "만족도(상/중/하)": "" # 직접 기입용
        })

# 데이터프레임 저장
df = pd.DataFrame(evaluation_data)
output_file = "model_evaluation.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\n✅ 평가 데이터 생성이 완료되었습니다: {output_file}")

🚀 테스트 시작 (총 100건)
📍 [1/10] 20대 자기계발생 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [00:54<00:00,  5.49s/it]


📍 [2/10] 20대 사회초년생 자취형 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:05<00:00,  6.57s/it]


📍 [3/10] 20대 외출 활동형 시나리오 진행 중...


...

📍 [9/10] 40대 교육비 집중형 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:45<00:00, 10.52s/it]


📍 [10/10] 디지털 구독 소비형 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:47<00:00, 10.71s/it]


✅ 평가 데이터 생성이 완료되었습니다: model_evaluation.csv


# 프롬프트 수정

In [7]:
import os
import json
import pandas as pd
import getpass
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from tqdm import tqdm

# ==========================================
# 1. 초기 설정 및 보안 인증
# ==========================================
MY_API_KEY = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)

# 경로 설정: evaluation 폴더 기준 상위 폴더의 data 참조
CHROMA_PATH = "../data/test/card_vector_db"

# Embedding 함수 설정
ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# Vector DB 연결
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name="card_separated_collection",
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 출력 형식 가이드 정의 (7가지 항목)
# ==========================================
FORMAT_GUIDE = """
반드시 아래 형식을 엄격히 지켜서 답변하십시오. 항목 외의 불필요한 말은 하지 마세요.

1. 카드명: 
2. 카드사: 
3. 카드종류: (체크 또는 신용)
4. 전월실적: 
5. 혜택: 
6. 해외사용: 
7. 추천 이유: 
"""

# ==========================================
# 3. 모델 응답 함수 정의
# ==========================================

# (1) Base GPT-3.5-turbo 응답 함수
def get_base_model_response(query):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": f"당신은 금융 상품 추천 전문가입니다. {FORMAT_GUIDE}"},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# (2) RAG 구성 후 응답 함수
def get_rag_model_response(query, persona_dict, target_card_type=None):
    persona_traits = persona_dict['traits']
    search_query = f"{persona_traits} {query}"
    
    # DB 검색 (ChromaDB 필터링 적용)
    if target_card_type in ["체크", "신용"]:
        results = collection.query(
            query_texts=[search_query], 
            n_results=15, 
            where={"card_type": target_card_type}
        )
    else:
        results = collection.query(query_texts=[search_query], n_results=15)

    raw_documents = results['documents'][0]
    available_card_names = set()
    card_context_map = {}
    
    # 검색된 데이터 정리
    for doc in raw_documents:
        first_line = doc.split("\n")[0]
        card_title = first_line.split("카드명: ")[1].split("|")[0].strip()
        available_card_names.add(card_title)
        if card_title not in card_context_map: card_context_map[card_title] = []
        card_context_map[card_title].append(doc)

    verified_list_str = ", ".join(list(available_card_names))
    formatted_context = ""
    for title, contents in card_context_map.items():
        formatted_context += f"\n### 카드 데이터: {title} ###\n" + "\n".join(contents) + "\n"

    # RAG 전용 프롬프트 구성
    system_prompt = f"""
    당신은 금융 상품 추천 전문가입니다. 반드시 아래의 [검증된 카드 리스트]와 [상세 데이터]에 있는 정보로만 답변하십시오.
    
    {FORMAT_GUIDE}

    [검증된 카드 리스트]: {verified_list_str}
    [상세 데이터]: {formatted_context}
    [사용자 페르소나]: {persona_dict['name']} ({persona_traits})
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# ==========================================
# 4. 테스트 시나리오 설정
# ==========================================
test_scenarios = [
    {"persona": {"name": "20대 자기계발생", "traits": "무실적 카드 선호, 대중교통 및 학원 이용 잦음"}, "query": "전월 실적 부담 없으면서 대중교통, 학원 혜택이 좋은 체크카드 알려줘.", "target": "체크"},
    {"persona": {"name": "20대 사회초년생 자취형", "traits": "배달·카페·온라인 쇼핑 비중 높음"}, "query": "배달 음식과 카페, 온라인 쇼핑에서 자주 쓰기 좋은 체크카드 추천해줘.", "target": "체크"},
    {"persona": {"name": "20대 외출 활동형", "traits": "외식과 카페 이용 빈도 높음, 문화생활"}, "query": "카페와 외식 위주로 자주 쓰기 좋은 체크카드를 알려줘.", "target": "체크"},
    {"persona": {"name": "30대 출퇴근 직장인", "traits": "대중교통 출퇴근, 점심 외식과 커피 지출"}, "query": "출퇴근 교통비랑 회사 근처 카페 혜택이 집중된 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 자차 보유 직장인", "traits": "자차 출퇴근, 주유비 지출 큼"}, "query": "주유 혜택이 크고 자차 이용자에게 도움이 되는 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 여행 중심형", "traits": "해외 여행 및 직구 빈번, 항공권·숙박"}, "query": "해외 결제 수수료 혜택이 좋고 여행 시 활용하기 좋은 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 건강관리 소비형", "traits": "병원, 약국, 헬스장 이용 빈도 높음"}, "query": "병원이나 헬스장 지출에 혜택을 받을 수 있는 신용카드를 알려줘.", "target": "신용"},
    {"persona": {"name": "40대 가족 생활형", "traits": "대형마트 장보기, 생활비와 고정비 관리"}, "query": "대형마트 장보기랑 생활비 지출에 혜택이 좋은 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "40대 교육비 집중형", "traits": "자녀 학원비 결제 비중 큼"}, "query": "자녀 학원비 결제 시 혜택을 받을 수 있는 신용카드를 알려줘.", "target": "신용"},
    {"persona": {"name": "디지털 구독 소비형", "traits": "OTT, 온라인 쇼핑, 구독 서비스"}, "query": "온라인 결제나 구독 서비스에서 할인/적립이 있는 체크카드 추천해줘.", "target": "체크"}
]

# ==========================================
# 5. 실행 및 결과 저장
# ==========================================
ITERATIONS = 1 # 각 시나리오별 10회 반복 (총 100회)
evaluation_data = []

print(f"🚀 비교 평가 시작 (총 {len(test_scenarios) * ITERATIONS}건)")

for i, sc in enumerate(test_scenarios):
    print(f"📍 [{i+1}/10] {sc['persona']['name']} 테스트 중...")
    
    for run in tqdm(range(1, ITERATIONS + 1), desc="반복"):
        base_ans = get_base_model_response(sc['query'])
        rag_ans = get_rag_model_response(sc['query'], sc['persona'], target_card_type=sc['target'])
        
        evaluation_data.append({
            "ID": f"{i+1}-{run}",
            "페르소나": sc['persona']['name'],
            "질문": sc['query'],
            "Base_GPT_응답": base_ans,
            "RAG_시스템_응답": rag_ans,
            "Human_Fact_Check": "", # 사람이 직접 채울 공간
            "만족도(상/중/하)": ""    # 사람이 직접 채울 공간
        })

# CSV 저장 (utf-8-sig로 저장해야 엑셀에서 한글이 깨지지 않습니다)
df = pd.DataFrame(evaluation_data)
output_path = "test.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n✅ 평가 완료! 파일이 생성되었습니다: {output_path}")

🚀 비교 평가 시작 (총 10건)
📍 [1/10] 20대 자기계발생 테스트 중...


반복: 100%|██████████| 1/1 [00:12<00:00, 12.02s/it]


📍 [2/10] 20대 사회초년생 자취형 테스트 중...


반복: 100%|██████████| 1/1 [00:09<00:00,  9.27s/it]


📍 [3/10] 20대 외출 활동형 테스트 중...


...

📍 [9/10] 40대 교육비 집중형 테스트 중...


반복: 100%|██████████| 1/1 [00:31<00:00, 31.38s/it]


📍 [10/10] 디지털 구독 소비형 테스트 중...


반복: 100%|██████████| 1/1 [00:12<00:00, 12.60s/it]


✅ 평가 완료! 파일이 생성되었습니다: test.csv


# 가장 적합한 카드 1개만 추천하는 평가 코드

In [16]:
import os
import json
import pandas as pd
import getpass
import chromadb
import sys
import io
from chromadb.utils import embedding_functions
from openai import OpenAI
from tqdm import tqdm

# [추가] 시스템 출력 인코딩 설정 (한글 깨짐 및 ASCII 에러 방지)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

# ==========================================
# 1. 초기 설정 및 보안 인증
# ==========================================
MY_API_KEY = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)

# 경로 설정: evaluation 폴더 기준 상위 폴더의 data 참조
CHROMA_PATH = "../data/test/card_vector_db"

# Embedding 함수 설정
ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# Vector DB 연결
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name="card_separated_collection",
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 출력 형식 가이드 (단 하나만 추천하도록 제약 강화)
# ==========================================
FORMAT_GUIDE = """
[주의사항]
- 여러 개의 카드를 추천하지 마십시오. 반드시 가장 적합한 '단 하나의 카드'만 선정하여 답변하십시오.
- 아래 형식을 엄격히 지켜주시고, 형식 외의 인사말이나 불필요한 서술은 절대 하지 마십시오.

[형식]
1. 카드명: 
2. 카드사: 
3. 카드종류: (체크 또는 신용)
4. 전월실적: 
5. 혜택: 
6. 해외사용: 
7. 추천 이유: 
"""

# ==========================================
# 3. 모델 응답 함수 정의
# ==========================================

# (1) Base GPT-3.5-turbo 응답 함수
def get_base_model_response(query):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": f"당신은 금융 상품 추천 전문가입니다. {FORMAT_GUIDE}"},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# (2) RAG 구성 후 응답 함수
def get_rag_model_response(query, persona_dict, target_card_type=None):
    persona_traits = persona_dict['traits']
    search_query = f"{persona_traits} {query}"
    
    # DB 검색
    if target_card_type in ["체크", "신용"]:
        results = collection.query(
            query_texts=[search_query], 
            n_results=15, 
            where={"card_type": target_card_type}
        )
    else:
        results = collection.query(query_texts=[search_query], n_results=15)

    raw_documents = results['documents'][0]
    available_card_names = set()
    card_context_map = {}
    
    for doc in raw_documents:
        first_line = doc.split("\n")[0]
        card_title = first_line.split("카드명: ")[1].split("|")[0].strip()
        available_card_names.add(card_title)
        if card_title not in card_context_map: card_context_map[card_title] = []
        card_context_map[card_title].append(doc)

    verified_list_str = ", ".join(list(available_card_names))
    formatted_context = ""
    for title, contents in card_context_map.items():
        formatted_context += f"\n### 카드 데이터: {title} ###\n" + "\n".join(contents) + "\n"

    system_prompt = f"""
        당신은 초개인화 금융 상품 큐레이터입니다. 
        사용자의 연령대, 소비 패턴, 고정 지출, 특별한 선호도(예: 무실적 선호)를 분석하여 
        제공된 데이터 내에서 가장 완벽하게 어울리는 카드 '단 1개'를 추천하십시오.

        [분석 가이드라인]
        1. 실적 조건 확인: 사용자가 '무실적'이나 '실적 부담 없음'을 언급했다면 전월 실적 조건이 없거나 낮은 상품을 우선하십시오.
        2. 연령 및 라이프스타일 매칭: 대학생(체크/저가커피/교통), 직장인(신용/주유/점심), 학부모(교육/마트) 등 핵심 카테고리를 확인하십시오.
        3. 근거의 명확성: 추천 이유 작성 시, 페르소나의 어떤 특성(예: '카공족')이 해당 카드의 어떤 혜택(예: '스타벅스 50% 할인')과 연결되는지 구체적으로 서술하십시오.

        [검증된 카드 리스트]
        {verified_list_str}

        [상세 데이터 정보]
        {formatted_context}

        [사용자 페르소나]
        - 대상: {persona_dict['name']}
        - 세부 특징: {persona_dict['traits']}

        ⛔ 답변 형식 (이 양식 외에 인사말이나 꼬리말은 절대 금지):
        1. 카드명: 
        2. 카드사: 
        3. 카드종류: (체크 또는 신용)
        4. 전월실적: 
        5. 혜택: 
        6. 해외사용: 
        7. 추천 이유: 
        """

    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# ==========================================
# 4. 테스트 시나리오 및 실행 (이전과 동일)
# ==========================================
test_scenarios = [
    {
        "persona": {
            "name": "20대 알뜰 대학생(취준생)", 
            "traits": "수입이 불규칙함. 전월 실적 채우기 부담스러워함(무실적 선호). 대중교통(버스/지하철) 매일 이용. 카공족이라 스타디카페 및 저가 커피(메가, 빽다방) 자주 방문. 편의점 도시락으로 식사 해결 잦음."
        }, 
        "query": "실적 안 채워도 대중교통이랑 커피, 편의점 할인이 많이 되는 체크카드 추천해줘. 대학생이라 연회비 없는 게 좋아.", 
        "target": "체크"
    },
    {
        "persona": {
            "name": "20대 사회초년생 자취러", 
            "traits": "첫 월급 관리 시작. 배달의민족/쿠팡이츠 매주 3회 이상 이용. 쿠팡 와우 멤버십 및 네이버플러스 멤버십 구독 중. 자취 고정비(전기/가스/수도) 절약 희망."
        }, 
        "query": "배달 앱이랑 온라인 쇼핑 혜택이 제일 큰 체크카드 알려줘. 자취하면서 나가는 고정비도 조금이라도 환급받고 싶어.", 
        "target": "체크"
    },
    {
        "persona": {
            "name": "20대 힙스터 외출형", 
            "traits": "성수, 한남동 등 핫플레이스 맛집 탐방이 취미. 전시회, 영화 관람 등 문화생활 즐김. 친구들과 더치페이/소액 결제가 잦음. 디자인이 예쁜 카드를 선호함."
        }, 
        "query": "맛집이나 카페, 영화관에서 할인 많이 되는 체크카드 있어? 소액 결제할 때마다 포인트 쌓이는 게 체감이 컸으면 좋겠어.", 
        "target": "체크"
    },
    {
        "persona": {
            "name": "30대 광역버스 출퇴근러", 
            "traits": "경기도에서 서울로 매일 출퇴근. 교통비 지출 월 10만 원 이상. 점심은 회사 근처 식당 이용 후 프랜차이즈 카페(스타벅스/투썸) 방문. 패턴이 일정함."
        }, 
        "query": "광역버스나 지하철 교통비 혜택이 가장 큰 신용카드 추천해줘. 매일 먹는 점심 식대랑 스타벅스 할인도 꼭 포함되면 좋겠어.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "30대 자차 보유 프로운전러", 
            "traits": "영업직이라 자차 이동 많음. 주유비 지출이 월 30만 원 이상. 하이패스, 주차장, 자동차 보험료 등 차량 유지비 관리에 집중함."
        }, 
        "query": "리터당 주유 할인이나 포인트 적립이 제일 많이 되는 신용카드 알려줘. 주차장 할인이나 정비소 혜택도 같이 있으면 좋아.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "30대 여행 매니아(직구족)", 
            "traits": "연 2~3회 해외 여행. 평소 알리, 테무, 아마존 등 해외 직구 자주 이용. 항공 마일리지 적립보다 당장 체감되는 결제 수수료 면제나 라운지 이용 선호."
        }, 
        "query": "해외 결제 수수료가 없고 라운지 무료 이용이 가능한 여행 특화 신용카드 추천해줘. 직구할 때도 적립되는 카드가 필요해.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "30대 갓생 건강러", 
            "traits": "주 4회 헬스장 출석. 필라테스 및 요가 수강 중. 아이허브에서 영양제 정기 구매. 정기적인 치과/피부과 진료로 병원비 지출이 꾸준함."
        }, 
        "query": "헬스장이나 필라테스 같은 운동 시설, 그리고 병원/약국에서 혜택이 집중된 신용카드 알려줘. 건강기능식품 구매 혜택도 궁금해.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "40대 생활비 만렙 주부", 
            "traits": "이마트, 홈플러스 등 대형마트에서 주말마다 장보기. 아파트 관리비, 가스비, 통신비 등 온 가족 고정비를 한 카드로 몰아서 결제함."
        }, 
        "query": "마트 장보기 할인율이 높고 아파트 관리비랑 공과금 결제할 때 혜택을 받을 수 있는 생활비 신용카드 추천해줘.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "40대 교육열 부모님", 
            "traits": "중/고등학생 자녀 둔 학부모. 매월 대형 학원 및 학습지 결제 금액이 큼. 자녀 간식용 편의점 결제나 주말 외식 비중이 높음."
        }, 
        "query": "학원비 결제 시 할인 한도가 큰 신용카드 위주로 알려줘. 교육비 결제 금액이 커서 실적 채우기는 쉬운 편이야.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "2030 디지털 노마드(구독러)", 
            "traits": "유튜브 프리미엄, 넷플릭스, 티빙 등 OTT 3개 이상 구독. 웹툰과 게임 인앱 결제 잦음. 주로 무신사나 지그재그에서 패션 쇼핑함."
        }, 
        "query": "OTT 구독료랑 무신사 같은 쇼핑몰에서 혜택이 좋은 체크카드 추천해줘. 배달이나 편의점도 자주 이용해.", 
        "target": "체크"
    }
]
ITERATIONS = 10
evaluation_data = []

print(f"🚀 1:1 비교 평가 시작 (총 {len(test_scenarios) * ITERATIONS}건)")

for i, sc in enumerate(test_scenarios):
    print(f"📍 [{i+1}/10] {sc['persona']['name']} 시나리오 진행 중...")
    
    for run in tqdm(range(1, ITERATIONS + 1), desc="반복 회차"):
        base_ans = get_base_model_response(sc['query'])
        rag_ans = get_rag_model_response(sc['query'], sc['persona'], target_card_type=sc['target'])
        
        evaluation_data.append({
            "시나리오_ID": i + 1,
            "회차": run,
            "페르소나": sc['persona']['name'],
            "질문": sc['query'],
            "Base_GPT_응답": base_ans,
            "RAG_시스템_응답": rag_ans,
            "Human_Evaluation": "", 
            "만족도(상/중/하)": "" 
        })

df = pd.DataFrame(evaluation_data)
df.to_csv("model_evaluation_final.csv", index=False, encoding="utf-8-sig")
print("\n✅ 평가 데이터 생성이 완료되었습니다: model_evaluation_single.csv")

🚀 1:1 비교 평가 시작 (총 100건)
📍 [1/10] 20대 알뜰 대학생(취준생) 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:01<00:00,  6.17s/it]


📍 [2/10] 20대 사회초년생 자취러 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:06<00:00,  6.66s/it]


📍 [3/10] 20대 힙스터 외출형 시나리오 진행 중...


...

📍 [9/10] 40대 교육열 부모님 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [00:56<00:00,  5.61s/it]


📍 [10/10] 2030 디지털 노마드(구독러) 시나리오 진행 중...


반복 회차: 100%|██████████| 10/10 [01:07<00:00,  6.78s/it]


✅ 평가 데이터 생성이 완료되었습니다: model_evaluation_single.csv


# iteration 오류(?) 수정

In [17]:
import os
import json
import pandas as pd
import getpass
import chromadb
import sys
import random
from chromadb.utils import embedding_functions
from openai import OpenAI
from tqdm import tqdm

# ==========================================
# 0. 출력 인코딩 설정
# ==========================================
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

# ==========================================
# 1. 인증 및 환경 설정
# ==========================================
MY_API_KEY = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)

CHROMA_PATH = "../data/test/card_vector_db"

ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name="card_separated_collection",
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 출력 형식 가이드
# ==========================================
FORMAT_GUIDE = """
⛔ 아래 형식 외의 출력은 허용되지 않습니다.

1. 카드명:
2. 카드사:
3. 카드종류: (체크 또는 신용)
4. 전월실적:
5. 혜택:
6. 해외사용:
7. 추천 이유:
"""

# ==========================================
# 3. Base 모델 응답
# ==========================================
def get_base_model_response(query):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": f"당신은 금융 상품 추천 전문가입니다. {FORMAT_GUIDE}"
            },
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.4  # 🔹 iteration별 변동 가능
    )
    return response.choices[0].message.content

# ==========================================
# 4. RAG 모델 응답 (ITERATION 반영)
# ==========================================
def get_rag_model_response(query, persona_dict, run, target_card_type=None):

    persona_traits = persona_dict["traits"]

    # 🔹 iteration마다 다른 embedding 위치 유도
    search_query = f"{persona_traits} {query} iteration-{run}"

    # 🔹 retrieval 개수 랜덤화
    n_results = random.choice([8, 10, 12, 15])

    if target_card_type in ["체크", "신용"]:
        results = collection.query(
            query_texts=[search_query],
            n_results=n_results,
            where={"card_type": target_card_type}
        )
    else:
        results = collection.query(
            query_texts=[search_query],
            n_results=n_results
        )

    raw_documents = results["documents"][0]

    card_context_map = {}
    available_card_names = set()

    for doc in raw_documents:
        first_line = doc.split("\n")[0]
        card_title = first_line.split("카드명: ")[1].split("|")[0].strip()

        available_card_names.add(card_title)
        card_context_map.setdefault(card_title, []).append(doc)

    verified_list_str = ", ".join(list(available_card_names))

    formatted_context = ""
    for title, contents in card_context_map.items():
        formatted_context += f"\n### 카드 데이터: {title} ###\n"
        formatted_context += "\n".join(contents) + "\n"

    system_prompt = f"""
당신은 초개인화 금융 상품 큐레이터입니다.

- 여러 카드 후보를 비교 분석하십시오.
- 최종 출력은 반드시 단 하나의 카드만 추천하십시오.
- 반드시 제공된 카드 데이터 내에서만 선택하십시오.

[검증된 카드 리스트]
{verified_list_str}

[카드 상세 데이터]
{formatted_context}

[사용자 페르소나]
- 이름: {persona_dict["name"]}
- 특징: {persona_traits}

{FORMAT_GUIDE}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.2  # 🔹 RAG는 안정성 유지
    )

    return response.choices[0].message.content

# ==========================================
# 5. 테스트 시나리오
# ==========================================
test_scenarios = [
    {
        "persona": {
            "name": "20대 알뜰 대학생(취준생)", 
            "traits": "수입이 불규칙함. 전월 실적 채우기 부담스러워함(무실적 선호). 대중교통(버스/지하철) 매일 이용. 카공족이라 스타디카페 및 저가 커피(메가, 빽다방) 자주 방문. 편의점 도시락으로 식사 해결 잦음."
        }, 
        "query": "실적 안 채워도 대중교통이랑 커피, 편의점 할인이 많이 되는 체크카드 추천해줘. 대학생이라 연회비 없는 게 좋아.", 
        "target": "체크"
    },
    {
        "persona": {
            "name": "20대 사회초년생 자취러", 
            "traits": "첫 월급 관리 시작. 배달의민족/쿠팡이츠 매주 3회 이상 이용. 쿠팡 와우 멤버십 및 네이버플러스 멤버십 구독 중. 자취 고정비(전기/가스/수도) 절약 희망."
        }, 
        "query": "배달 앱이랑 온라인 쇼핑 혜택이 제일 큰 체크카드 알려줘. 자취하면서 나가는 고정비도 조금이라도 환급받고 싶어.", 
        "target": "체크"
    },
    {
        "persona": {
            "name": "20대 힙스터 외출형", 
            "traits": "성수, 한남동 등 핫플레이스 맛집 탐방이 취미. 전시회, 영화 관람 등 문화생활 즐김. 친구들과 더치페이/소액 결제가 잦음. 디자인이 예쁜 카드를 선호함."
        }, 
        "query": "맛집이나 카페, 영화관에서 할인 많이 되는 체크카드 있어? 소액 결제할 때마다 포인트 쌓이는 게 체감이 컸으면 좋겠어.", 
        "target": "체크"
    },
    {
        "persona": {
            "name": "30대 광역버스 출퇴근러", 
            "traits": "경기도에서 서울로 매일 출퇴근. 교통비 지출 월 10만 원 이상. 점심은 회사 근처 식당 이용 후 프랜차이즈 카페(스타벅스/투썸) 방문. 패턴이 일정함."
        }, 
        "query": "광역버스나 지하철 교통비 혜택이 가장 큰 신용카드 추천해줘. 매일 먹는 점심 식대랑 스타벅스 할인도 꼭 포함되면 좋겠어.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "30대 자차 보유 프로운전러", 
            "traits": "영업직이라 자차 이동 많음. 주유비 지출이 월 30만 원 이상. 하이패스, 주차장, 자동차 보험료 등 차량 유지비 관리에 집중함."
        }, 
        "query": "리터당 주유 할인이나 포인트 적립이 제일 많이 되는 신용카드 알려줘. 주차장 할인이나 정비소 혜택도 같이 있으면 좋아.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "30대 여행 매니아(직구족)", 
            "traits": "연 2~3회 해외 여행. 평소 알리, 테무, 아마존 등 해외 직구 자주 이용. 항공 마일리지 적립보다 당장 체감되는 결제 수수료 면제나 라운지 이용 선호."
        }, 
        "query": "해외 결제 수수료가 없고 라운지 무료 이용이 가능한 여행 특화 신용카드 추천해줘. 직구할 때도 적립되는 카드가 필요해.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "30대 갓생 건강러", 
            "traits": "주 4회 헬스장 출석. 필라테스 및 요가 수강 중. 아이허브에서 영양제 정기 구매. 정기적인 치과/피부과 진료로 병원비 지출이 꾸준함."
        }, 
        "query": "헬스장이나 필라테스 같은 운동 시설, 그리고 병원/약국에서 혜택이 집중된 신용카드 알려줘. 건강기능식품 구매 혜택도 궁금해.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "40대 생활비 만렙 주부", 
            "traits": "이마트, 홈플러스 등 대형마트에서 주말마다 장보기. 아파트 관리비, 가스비, 통신비 등 온 가족 고정비를 한 카드로 몰아서 결제함."
        }, 
        "query": "마트 장보기 할인율이 높고 아파트 관리비랑 공과금 결제할 때 혜택을 받을 수 있는 생활비 신용카드 추천해줘.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "40대 교육열 부모님", 
            "traits": "중/고등학생 자녀 둔 학부모. 매월 대형 학원 및 학습지 결제 금액이 큼. 자녀 간식용 편의점 결제나 주말 외식 비중이 높음."
        }, 
        "query": "학원비 결제 시 할인 한도가 큰 신용카드 위주로 알려줘. 교육비 결제 금액이 커서 실적 채우기는 쉬운 편이야.", 
        "target": "신용"
    },
    {
        "persona": {
            "name": "2030 디지털 노마드(구독러)", 
            "traits": "유튜브 프리미엄, 넷플릭스, 티빙 등 OTT 3개 이상 구독. 웹툰과 게임 인앱 결제 잦음. 주로 무신사나 지그재그에서 패션 쇼핑함."
        }, 
        "query": "OTT 구독료랑 무신사 같은 쇼핑몰에서 혜택이 좋은 체크카드 추천해줘. 배달이나 편의점도 자주 이용해.", 
        "target": "체크"
    }
]

# ==========================================
# 6. 평가 실행
# ==========================================
ITERATIONS = 10
evaluation_data = []

print(f"🚀 평가 시작 (총 {len(test_scenarios) * ITERATIONS}회)")

for i, sc in enumerate(test_scenarios):
    print(f"\n📍 [{i+1}] {sc['persona']['name']}")

    for run in tqdm(range(1, ITERATIONS + 1), desc="ITERATION"):
        base_ans = get_base_model_response(sc["query"])
        rag_ans = get_rag_model_response(
            sc["query"],
            sc["persona"],
            run=run,
            target_card_type=sc["target"]
        )

        evaluation_data.append({
            "시나리오_ID": i + 1,
            "ITERATION": run,
            "페르소나": sc["persona"]["name"],
            "질문": sc["query"],
            "Base_GPT_응답": base_ans,
            "RAG_시스템_응답": rag_ans,
            "Human_Evaluation": "",
            "만족도(상/중/하)": ""
        })

# ==========================================
# 7. 결과 저장
# ==========================================
df = pd.DataFrame(evaluation_data)
df.to_csv("model_evaluation_iteration_enabled.csv", index=False, encoding="utf-8-sig")

print("\n✅ 평가 데이터 생성 완료: model_evaluation_iteration_enabled.csv")


🚀 평가 시작 (총 100회)

📍 [1] 20대 알뜰 대학생(취준생)


ITERATION: 100%|██████████| 10/10 [01:01<00:00,  6.12s/it]



📍 [2] 20대 사회초년생 자취러


ITERATION: 100%|██████████| 10/10 [00:55<00:00,  5.50s/it]



📍 [3] 20대 힙스터 외출형


...


📍 [9] 40대 교육열 부모님


ITERATION: 100%|██████████| 10/10 [00:59<00:00,  5.92s/it]



📍 [10] 2030 디지털 노마드(구독러)


ITERATION: 100%|██████████| 10/10 [01:13<00:00,  7.34s/it]


✅ 평가 데이터 생성 완료: model_evaluation_iteration_enabled.csv
